# Preprocessing & Stratified Merging of Datasets for Image Restoration

This notebook recursively scans all subfolders (`train`, `test`, `val`, etc.) across `BSD500`, `CIFAR-10`, `DIV2K`, and `CelebA` datasets.
It splits every individual dataset proportionally into **80% Train / 10% Validation / 10% Test** splits so that all 3 splits contain a balanced representation of face, landscape, object, and natural scene images.

Output directory structure:
```
/kaggle/working/clean_images/
├── train/
├── val/
└── test/
```

In [ ]:
import os
import glob
import random
from pathlib import Path
from PIL import Image
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# Set random seed for reproducible train/val/test splits
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# 1. Environment & Path Setup
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    OUTPUT_BASE = '/kaggle/working/clean_images'
    DATASET_PATHS = {
        'bsd500': '/kaggle/input/bsd500original',
        'cifar10': '/kaggle/input/cifar10',
        'div2k': '/kaggle/input/div2k-dataset',
        'celeba': '/kaggle/input/celeba-dataset'
    }
else:
    OUTPUT_BASE = '../data/clean_images'
    DATASET_PATHS = {
        'bsd500': '../data/bsd500',
        'cifar10': '../data/cifar10',
        'div2k': '../data/div2k',
        'celeba': '../data/celeba'
    }

# Train / Val / Test Split Ratios
SPLIT_RATIOS = {'train': 0.80, 'val': 0.10, 'test': 0.10}

# Create Train, Val, Test Output Subdirectories
SPLIT_DIRS = {}
for split in ['train', 'val', 'test']:
    split_path = os.path.join(OUTPUT_BASE, split)
    os.makedirs(split_path, exist_ok=True)
    SPLIT_DIRS[split] = split_path

print(f"Running on Kaggle: {IS_KAGGLE}")
print(f"Output Base Directory: {OUTPUT_BASE}")
print(f"Created Split Subdirectories: {list(SPLIT_DIRS.keys())}")

In [ ]:
# 2. Dynamic Recursive Image Discovery across ALL Subfolders (train, val, test)
def find_all_images(base_path, ds_name="", extensions=('.jpg', '.jpeg', '.png', '.bmp', '.webp')):
    """
    Recursively scans base_path and ALL subfolders (train, test, val, etc.)
    to gather all available image files. Performs dynamic path resolution on Kaggle if needed.
    """
    image_paths = []
    if not os.path.exists(base_path):
        print(f"Path not directly found: '{base_path}'. Attempting dynamic search...")
        if IS_KAGGLE:
            query = ds_name.lower() if ds_name else os.path.basename(base_path).lower()
            for root, dirs, files in os.walk('/kaggle/input'):
                if query in root.lower():
                    base_path = root
                    print(f" -> Dynamically resolved path to: {base_path}")
                    break
    
    if os.path.exists(base_path):
        for root, dirs, files in os.walk(base_path):
            for file in files:
                if file.lower().endswith(extensions):
                    image_paths.append(os.path.join(root, file))
    return image_paths

# Collect all image paths for every dataset
dataset_files = {}
for ds_name, ds_path in DATASET_PATHS.items():
    files = find_all_images(ds_path, ds_name=ds_name)
    random.shuffle(files)  # Shuffle deterministically
    dataset_files[ds_name] = files
    print(f"Found Total {len(files)} images across ALL subfolders in '{ds_name}'")

In [ ]:
# 3. Preprocessing Function (Crop + Bicubic Resize to 128x128 RGB PNG)
TARGET_SIZE = (128, 128)  # Target uniform resolution

def preprocess_and_save(img_input, save_path, target_size=TARGET_SIZE):
    """
    Converts image to RGB, center-crops to square aspect ratio,
    resizes with high-quality Bicubic interpolation, and saves as PNG.
    """
    try:
        if isinstance(img_input, str):
            with Image.open(img_input) as img:
                img = img.convert('RGB')
                w, h = img.size
                min_dim = min(w, h)
                left = (w - min_dim) // 2
                top = (h - min_dim) // 2
                img_cropped = img.crop((left, top, left + min_dim, top + min_dim))
                img_resized = img_cropped.resize(target_size, Image.Resampling.BICUBIC)
                img_resized.save(save_path, 'PNG')
                return True
        elif isinstance(img_input, Image.Image):
            img = img_input.convert('RGB')
            w, h = img.size
            min_dim = min(w, h)
            left = (w - min_dim) // 2
            top = (h - min_dim) // 2
            img_cropped = img.crop((left, top, left + min_dim, top + min_dim))
            img_resized = img_cropped.resize(target_size, Image.Resampling.BICUBIC)
            img_resized.save(save_path, 'PNG')
            return True
    except Exception as e:
        return False

In [ ]:
# 4. Stratified Proportionated Dataset Splitting (80% Train / 10% Val / 10% Test)
MAX_SAMPLES = {
    'bsd500': 1000,    # Process all BSD500 images
    'div2k': 2000,     # Process DIV2K images
    'celeba': 15000,   # Balanced subset of CelebA faces
    'cifar10': 5000    # CIFAR-10 sample subset
}

stats = {'train': 0, 'val': 0, 'test': 0}

for ds_name, file_list in dataset_files.items():
    limit = MAX_SAMPLES.get(ds_name, 5000)
    selected_files = file_list[:limit] if limit else file_list
    n_total = len(selected_files)
    
    if n_total == 0:
        continue
        
    # Calculate exact split cutoffs
    n_train = int(n_total * SPLIT_RATIOS['train'])
    n_val = int(n_total * SPLIT_RATIOS['val'])
    
    train_files = selected_files[:n_train]
    val_files = selected_files[n_train:n_train + n_val]
    test_files = selected_files[n_train + n_val:]
    
    splits_data = {
        'train': train_files,
        'val': val_files,
        'test': test_files
    }
    
    print(f"\n--- Processing '{ds_name}' (Total: {n_total} -> Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}) ---")
    
    for split_name, files in splits_data.items():
        split_dir = SPLIT_DIRS[split_name]
        success_count = 0
        for idx, img_path in enumerate(tqdm(files, desc=f"{ds_name} -> {split_name}")):
            out_name = f"{ds_name}_{idx:06d}.png"
            out_path = os.path.join(split_dir, out_name)
            if preprocess_and_save(img_path, out_path, TARGET_SIZE):
                success_count += 1
        stats[split_name] += success_count

# Fallback for CIFAR-10 if raw image files were not found directly
if len(dataset_files.get('cifar10', [])) == 0:
    print("\nCIFAR-10 raw image files not found in dataset folders. Loading via torchvision datasets...")
    try:
        cifar_root = "/tmp/cifar10" if IS_KAGGLE else "../data/cifar10"
        cifar_ds = torchvision.datasets.CIFAR10(root=cifar_root, train=True, download=True)
        cifar_limit = MAX_SAMPLES.get('cifar10', 5000)
        indices = list(range(min(cifar_limit, len(cifar_ds))))
        random.shuffle(indices)
        
        n_total = len(indices)
        n_train = int(n_total * SPLIT_RATIOS['train'])
        n_val = int(n_total * SPLIT_RATIOS['val'])
        
        cifar_splits = {
            'train': indices[:n_train],
            'val': indices[n_train:n_train + n_val],
            'test': indices[n_train + n_val:]
        }
        
        for split_name, idx_list in cifar_splits.items():
            split_dir = SPLIT_DIRS[split_name]
            count = 0
            for i, idx in enumerate(tqdm(idx_list, desc=f"cifar10 -> {split_name}")):
                img, _ = cifar_ds[idx]
                out_path = os.path.join(split_dir, f"cifar10_{i:06d}.png")
                if preprocess_and_save(img, out_path, TARGET_SIZE):
                    count += 1
            stats[split_name] += count
    except Exception as e:
        print(f"CIFAR-10 torchvision fallback info: {e}")

print(f"\n==========================================")
print(f"DATASET PROCESSING COMPLETED SUCCESSFULLY!")
print(f"Train Images: {stats['train']}")
print(f"Val Images:   {stats['val']}")
print(f"Test Images:  {stats['test']}")
print(f"Total Clean Images: {sum(stats.values())}")
print(f"Saved Location: {OUTPUT_BASE}")
print(f"==========================================")

In [ ]:
# 5. Visualize Sample Preprocessed Images from Train, Val, and Test Splits
fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for split_idx, split_name in enumerate(['train', 'val', 'test']):
    split_path = SPLIT_DIRS[split_name]
    img_files = glob.glob(os.path.join(split_path, "*.png"))
    if len(img_files) > 0:
        samples = np.random.choice(img_files, min(5, len(img_files)), replace=False)
        for col_idx, img_path in enumerate(samples):
            ax = axes[split_idx, col_idx]
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"[{split_name.upper()}]\n{os.path.basename(img_path)[:20]}", fontsize=8)
            ax.axis('off')

plt.suptitle("Sample Preprocessed Clean Images Across Train / Val / Test Splits", fontsize=14)
plt.tight_layout()
plt.show()